# Cached Prediction EDA (HVG)

Streamlined companion notebook for HVG-scope cached analyses.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ar_utils.results_eda import *


In [ ]:
set_academic_style()

CFG = EDAConfig(
    csv_path='data/raw/gxp_samples.csv',
    hvg_path='data/raw/ahba_100hvg.txt',
    cache_root='out/loro_subject_cache',
    gene_scope='hvg',
)
CFG
print('analysis scope:', CFG.gene_scope)


## 1) Pre/Post ComBat Region Diagnostics


In [ ]:
PREPOST = prepare_pre_post_harmonization(CFG)
print('subjects:', len(PREPOST['subjects']))
print('genes:', len(PREPOST['genes']))
print('parcels:', PREPOST['raw_cube'].shape[1])
avail = available_regions(PREPOST, min_subjects=2)
display(avail.head(12))


In [ ]:
REGION = int(avail.iloc[0]['parcel_idx'])
fig, axes = plot_region_covariance_side_by_side(PREPOST, region=REGION, mode='subject')


Gene-wise covariance view is available via `mode='gene'` but omitted by default for speed.


## 2) Coverage vs Accuracy


In [ ]:
naive_df = compute_subject_metrics_from_cache(CFG, model='naive')
dlam_df = compute_subject_metrics_from_cache(CFG, model='dlam')
plam_df = compute_subject_metrics_from_cache(CFG, model='plam')
metrics_df = pd.concat([naive_df, dlam_df, plam_df], ignore_index=True)
display(metrics_df.head())


In [ ]:
fig, axes, summary_df = plot_loro_subject_summary_bars(metrics_df, use_sem=False)
display(summary_df)
fig, ax = plot_coverage_vs_accuracy(metrics_df, metric='pearson_r')
fig, ax = plot_coverage_vs_accuracy(metrics_df, metric='rmse')


## 3) Single-Subject Scatter Triplets


In [ ]:
fig, axes, subject = plot_single_subject_scatter_triplet(
    CFG,
    subject_mode='median',
    color_by='parcel',
    top_n=10,
)
print('subject:', subject)


In [ ]:
fig, axes, subject = plot_single_subject_scatter_triplet(
    CFG,
    subject_mode='median',
    color_by='gene',
    top_n=10,
)
print('subject:', subject)
